<a href="https://colab.research.google.com/github/meenakshi930/DiseaseLeafClassifier/blob/anshika-data/Anshika/notebooks/02_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Disease Leaf Classifier — Data Cleaning

## Objective

Clean and validate the PlantVillage dataset before preprocessing and model training.

### Cleaning Tasks
- Check for invalid or corrupted image files
- Check image readability
- Check for duplicate images
- Identify files that should be excluded from further processing
- Create a clean list of valid image files

In [1]:
import os
import hashlib
from PIL import Image

In [3]:
!pip -q install kagglehub

import kagglehub

path = kagglehub.dataset_download("emmarex/plantdisease")

print("Dataset downloaded to:")
print(path)

Using Colab cache for faster access to the 'plantdisease' dataset.
Dataset downloaded to:
/kaggle/input/plantdisease


In [4]:
print(os.listdir(path))

['PlantVillage', 'plantvillage']


In [5]:
plantvillage_path = os.path.join(path, "PlantVillage")

print("PlantVillage path:", plantvillage_path)
print("Exists:", os.path.exists(plantvillage_path))

PlantVillage path: /kaggle/input/plantdisease/PlantVillage
Exists: True


In [6]:
classes = sorted(os.listdir(plantvillage_path))

print("Number of classes:", len(classes))
print("\nClasses:")

for i, class_name in enumerate(classes, start=1):
    print(i, class_name)

Number of classes: 15

Classes:
1 Pepper__bell___Bacterial_spot
2 Pepper__bell___healthy
3 Potato___Early_blight
4 Potato___Late_blight
5 Potato___healthy
6 Tomato_Bacterial_spot
7 Tomato_Early_blight
8 Tomato_Late_blight
9 Tomato_Leaf_Mold
10 Tomato_Septoria_leaf_spot
11 Tomato_Spider_mites_Two_spotted_spider_mite
12 Tomato__Target_Spot
13 Tomato__Tomato_YellowLeaf__Curl_Virus
14 Tomato__Tomato_mosaic_virus
15 Tomato_healthy


In [7]:
print("Files in each class:\n")

for class_name in classes:
    class_path = os.path.join(plantvillage_path, class_name)
    file_count = len(os.listdir(class_path))

    print(f"{class_name}: {file_count}")

Files in each class:

Pepper__bell___Bacterial_spot: 997
Pepper__bell___healthy: 1478
Potato___Early_blight: 1000
Potato___Late_blight: 1000
Potato___healthy: 152
Tomato_Bacterial_spot: 2127
Tomato_Early_blight: 1000
Tomato_Late_blight: 1909
Tomato_Leaf_Mold: 952
Tomato_Septoria_leaf_spot: 1771
Tomato_Spider_mites_Two_spotted_spider_mite: 1676
Tomato__Target_Spot: 1404
Tomato__Tomato_YellowLeaf__Curl_Virus: 3209
Tomato__Tomato_mosaic_virus: 373
Tomato_healthy: 1591


In [8]:
invalid_files = []

for class_name in classes:
    class_path = os.path.join(plantvillage_path, class_name)

    for image_name in os.listdir(class_path):
        image_path = os.path.join(class_path, image_name)

        try:
            with Image.open(image_path) as image:
                image.verify()
        except:
            invalid_files.append((class_name, image_name))

print("Number of invalid files:", len(invalid_files))

print("\nInvalid files:")
for item in invalid_files:
    print(item)

Number of invalid files: 1

Invalid files:
('Tomato__Tomato_YellowLeaf__Curl_Virus', 'svn-r6Yb5c')


In [9]:
valid_files = []

for class_name in classes:
    class_path = os.path.join(plantvillage_path, class_name)

    for image_name in os.listdir(class_path):
        image_path = os.path.join(class_path, image_name)

        try:
            with Image.open(image_path) as image:
                image.verify()

            valid_files.append((class_name, image_name))

        except:
            pass

print("Total valid images:", len(valid_files))
print("Total invalid files:", len(invalid_files))

Total valid images: 20638
Total invalid files: 1


In [10]:
file_hashes = {}
duplicate_files = []

for class_name, image_name in valid_files:
    image_path = os.path.join(
        plantvillage_path,
        class_name,
        image_name
    )

    with open(image_path, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()

    if file_hash in file_hashes:
        duplicate_files.append(
            (class_name, image_name, file_hashes[file_hash])
        )
    else:
        file_hashes[file_hash] = (class_name, image_name)

print("Total valid images:", len(valid_files))
print("Unique images:", len(file_hashes))
print("Duplicate images:", len(duplicate_files))

Total valid images: 20638
Unique images: 20624
Duplicate images: 14


In [11]:
print("Duplicate files:\n")

for duplicate in duplicate_files:
    duplicate_class, duplicate_name, original = duplicate

    print("Duplicate :", duplicate_class, "/", duplicate_name)
    print("Original  :", original[0], "/", original[1])
    print()

Duplicate files:

Duplicate : Tomato_Late_blight / bd4f09bd-ee85-4ab1-bce0-8cde3fdd7f1b___GHLB_PS Leaf 23.7 Day 13.jpg
Original  : Tomato_Late_blight / 48c55974-9fe9-4f4b-94f7-c8cd127d1e05___GHLB_PS Leaf 23.7 Day 13.jpg

Duplicate : Tomato_Late_blight / e5d707cd-077c-43af-bda9-6138e516ff51___GHLB2 Leaf 8999.JPG
Original  : Tomato_Late_blight / 5f21282c-e2ef-4c4a-ace1-b5701fe7effc___GHLB2 Leaf 8999.JPG

Duplicate : Tomato_Late_blight / 5de6da85-f8c4-48c4-b463-3e6bd78884cc___GHLB_PS Leaf 24 Day 16.jpg
Original  : Tomato_Late_blight / 3fae9c64-18f0-4a67-9f97-554248bb1bed___GHLB_PS Leaf 24 Day 16.jpg

Duplicate : Tomato_Late_blight / c1775bad-7c02-41fb-bb7d-f8df91d60ac3___GHLB_PS Leaf 23.5 Day 13.jpg
Original  : Tomato_Late_blight / 98586693-fe1f-4ea0-8e27-4501b61cf09b___GHLB_PS Leaf 23.5 Day 13.jpg

Duplicate : Tomato_Late_blight / d6e6897a-5083-4914-9903-804c5684a956___GHLB2 Leaf 102.JPG
Original  : Tomato_Late_blight / d81682aa-746b-4e07-af2b-52ebb6f4c017___GHLB2 Leaf 102.JPG

Duplicate

In [12]:
seen_hashes = set()
clean_files = []

for class_name, image_name in valid_files:
    image_path = os.path.join(
        plantvillage_path,
        class_name,
        image_name
    )

    with open(image_path, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()

    if file_hash not in seen_hashes:
        seen_hashes.add(file_hash)
        clean_files.append((class_name, image_name))

print("Valid images:", len(valid_files))
print("Duplicate files removed from list:", len(valid_files) - len(clean_files))
print("Final clean images:", len(clean_files))

Valid images: 20638
Duplicate files removed from list: 14
Final clean images: 20624


In [13]:
import csv

clean_list_path = "/content/clean_image_list.csv"

with open(clean_list_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["class_name", "image_name"])

    for class_name, image_name in clean_files:
        writer.writerow([class_name, image_name])

print("Clean file list saved to:", clean_list_path)
print("Number of images:", len(clean_files))

Clean file list saved to: /content/clean_image_list.csv
Number of images: 20624


## Cleaning Summary

The PlantVillage dataset was checked for invalid files, corrupted images, and exact duplicate images.

### Results

| Cleaning Check | Result |
|---|---:|
| Total files | 20,639 |
| Valid images | 20,638 |
| Invalid/non-image files | 1 |
| Exact duplicate files | 14 |
| Final clean images | 20,624 |

### Invalid File

One invalid/non-image file was identified:

`Tomato__Tomato_YellowLeaf__Curl_Virus/svn-r6Yb5c`

This file will be excluded from further processing.

### Duplicate Images

14 exact duplicate images were identified using MD5 file hashes.

- 8 duplicates were found in `Tomato_Late_blight`.
- 6 duplicates were found in `Tomato_healthy`.
- No duplicates were found across different classes.

The duplicate files were excluded from the clean image list. The original dataset files were not deleted.

### Final Dataset

The resulting clean image list contains **20,624 unique, valid images across 15 classes**.

The clean image list has been saved as:

`clean_image_list.csv`

In [14]:
from google.colab import files

files.download("/content/clean_image_list.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>